In [47]:
# Baseline builder: encrypted_flow_only_ids (MAWI + CIC-IoT2023 + USTC benign)
# Paste into Jupyter or save as a .py and run.
# Requires: pandas, numpy, pyyaml
# If missing: pip install pandas numpy pyyaml python-dateutil

import os
import glob
import gzip
from datetime import datetime, timezone
from typing import Optional, List, Dict, Tuple
import numpy as np
import pandas as pd
import yaml

# ---------------- CONFIG ----------------
MAWI_GLOB  = r"E:/data/mawi/conn/*/conn.log.gz"
CIC_GLOB   = r"E:/encrypted flow/CIC-IoT2023/*/conn.log.gz"
USTC_GLOB  = r"E:/encrypted flow/USTC-TFC2016/*/conn.log.gz"

PROFILE_NAME   = "encrypted_flow_only_ids"
OUTPUT_DIR     = "baselines"
BINS_LOG10     = 50
BURST_WINDOW_S = 10
WINSOR         = (0.005, 0.995)
MIN_YEAR = 2005
MAX_YEAR = 2026
RNG_SEED = 42

# service alias map (used to canonicalize service labels)
_SERVICE_ALIAS = {
    "http-alt":"http", "www":"http", "mdns":"dns", "llmnr":"dns",
    "submission":"smtp", "pop3s":"pop3", "imaps":"imap", "ssl":"https",
    "https-alt":"https"
}

# ---------------- helpers ----------------
def _open_auto(path: str):
    return gzip.open(path, "rb") if path.endswith(".gz") else open(path, "rb")

def _peek_first_non_ws_char(path: str) -> Optional[str]:
    try:
        with _open_auto(path) as f:
            b = f.read(4096)
        s = b.decode("utf-8", errors="ignore")
        for ch in s:
            if not ch.isspace():
                return ch
    except Exception:
        return None
    return None

def _to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.astype(str).str.replace(",","").str.strip(), errors="coerce")

def _winsorize(series: pd.Series, q_low=WINSOR[0], q_high=WINSOR[1]) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").dropna().astype(float)
    if s.empty: return s
    lo, hi = s.quantile([q_low, q_high])
    return s.clip(lower=lo, upper=hi)

def _log10_hist_cdf(series: pd.Series, n_bins=BINS_LOG10):
    s = _winsorize(series)
    s = s[s > 0]
    if s.empty: return [], []
    logv = np.log10(s.to_numpy())
    hist, edges = np.histogram(logv, bins=n_bins, density=True)
    if np.sum(hist) == 0:
        return edges.tolist(), [0.0]*(len(edges)-1)
    cdf = np.cumsum(hist) / np.sum(hist)
    return edges.tolist(), cdf.tolist()

def _normalize_proto(v: str) -> str:
    v = str(v).strip().lower()
    if v in ("6","tcp"): return "tcp"
    if v in ("17","udp"): return "udp"
    if v in ("1","icmp"): return "icmp"
    return v or "unknown"

_PORT2SVC = {
    53:"dns", 80:"http", 443:"https", 123:"ntp", 20:"ftp-data", 21:"ftp", 22:"ssh",
    23:"telnet", 25:"smtp", 110:"pop3", 143:"imap", 69:"tftp", 161:"snmp",
    995:"pop3s", 993:"imaps", 587:"submission", 8080:"http-alt", 3306:"mysql",
    3389:"rdp", 5060:"sip", 5353:"mdns", 1900:"ssdp", 554:"rtsp", 8883:"mqtts", 1883:"mqtt"
}
def _infer_service_from_port(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    return s.map(lambda x: _PORT2SVC.get(int(x), None) if pd.notna(x) else None)

def canonicalize_service_for_baseline(series: pd.Series, service_aliases=_SERVICE_ALIAS) -> pd.Series:
    if series is None: return series
    out = series.astype(str).str.lower().map(lambda x: x if x not in ("-","none","nan","") else None)
    out = out.map(lambda x: service_aliases.get(x, x) if x is not None else None)
    return out.where(out.notna(), other=None)

# ---------------- Zeek/conn readers ----------------
def read_zeek_conn_any(path: str) -> pd.DataFrame:
    """Read Zeek conn.log (JSON lines or ASCII TSV with #fields)."""
    first = _peek_first_non_ws_char(path)
    try:
        with _open_auto(path) as f:
            if first == "{":
                df = pd.read_json(f, lines=True)
                return df
            # ASCII TSV with Zeek headers
            # find fields
            fobj = _open_auto(path)
            fields = None
            for raw in fobj:
                line = raw.decode("utf-8", errors="ignore").rstrip("\n")
                if line.startswith("#fields"):
                    parts = line.split("\t")
                    fields = parts[1:]
                    break
            fobj.close()
            if not fields:
                # fallback: read as TSV
                with _open_auto(path) as f2:
                    return pd.read_csv(f2, sep="\t", header=None, engine="python", comment="#")
            with _open_auto(path) as f3:
                df = pd.read_csv(f3, sep="\t", header=None, names=fields, engine="python", comment="#")
                return df
    except Exception as e:
        raise

def normalize_zeek_conn(df: pd.DataFrame) -> pd.DataFrame:
    cols = {str(c).lower(): c for c in df.columns}

    # bytes
    if "orig_bytes" in cols and "resp_bytes" in cols:
        b = _to_num(df[cols["orig_bytes"]]).fillna(0) + _to_num(df[cols["resp_bytes"]]).fillna(0)
    elif "bytes" in cols:
        b = _to_num(df[cols["bytes"]])
    else:
        b = pd.Series(np.nan, index=df.index)

    # pkts
    if "orig_pkts" in cols and "resp_pkts" in cols:
        p = _to_num(df[cols["orig_pkts"]]).fillna(0) + _to_num(df[cols["resp_pkts"]]).fillna(0)
    elif "pkts" in cols:
        p = _to_num(df[cols["pkts"]])
    elif "packets" in cols:
        p = _to_num(df[cols["packets"]])
    else:
        p = pd.Series(np.nan, index=df.index)

    # duration (Zeek: seconds float)
    d = _to_num(df[cols["duration"]]) if "duration" in cols else pd.Series(np.nan, index=df.index)

    # start timestamp
    if "ts" in cols:
        ts = _to_num(df[cols["ts"]])  # Zeek 'ts' is epoch seconds float
    elif "start" in cols:
        ts = _to_num(df[cols["start"]])
    else:
        ts = pd.Series(np.nan, index=df.index)

    proto = df[cols["proto"]].astype(str).map(_normalize_proto) if "proto" in cols else pd.Series("unknown", index=df.index)

    # service: prefer Zeek 'service' else infer from resp_p / id.resp_p
    service = None
    if "service" in cols:
        service = df[cols["service"]].astype(str).str.lower()
        if ("resp_p" in cols or "id.resp_p" in cols):
            resp_col = cols.get("resp_p") or cols.get("id.resp_p")
            inferred = _infer_service_from_port(df[resp_col])
            service = service.where(~service.isin(["-","none",""]), inferred)
    else:
        if ("resp_p" in cols or "id.resp_p" in cols):
            resp_col = cols.get("resp_p") or cols.get("id.resp_p")
            service = _infer_service_from_port(df[resp_col])
        else:
            service = None

    out = pd.DataFrame({
        "bytes": b,
        "pkts": p,
        "duration": d,
        "start_ts": ts,
        "proto": proto,
        "service": service if isinstance(service, pd.Series) else None,
    })
    return out.replace([np.inf, -np.inf], np.nan)

# ---------------- loaders for your three sources ----------------
def _match_paths(path_or_glob: str) -> List[str]:
    if os.path.isfile(path_or_glob):
        return [path_or_glob]
    return sorted(glob.glob(path_or_glob, recursive=True))

def load_encrypted_sources(mawi_glob: str, cic_glob: str, ustc_glob: str) -> pd.DataFrame:
    frames=[]
    # MAWI (Zeek conn logs)
    for p in _match_paths(mawi_glob):
        try:
            df_raw = read_zeek_conn_any(p)
            df = normalize_zeek_conn(df_raw)
            df["source_file"] = p
            frames.append(df)
            print(f"[OK] MAWI {p}: rows={len(df)}")
        except Exception as e:
            print(f"[WARN] MAWI {p}: {e}")

    # CIC (you produced Zeek conn.log.gz JSON-lines)
    for p in _match_paths(cic_glob):
        try:
            df_raw = read_zeek_conn_any(p)
            df = normalize_zeek_conn(df_raw)
            df["source_file"] = p
            frames.append(df)
            print(f"[OK] CIC-IoT CSV {p}: rows={len(df)}")
        except Exception as e:
            print(f"[WARN] CIC {p}: {e}")

    # USTC (Zeek conn logs)
    for p in _match_paths(ustc_glob):
        try:
            df_raw = read_zeek_conn_any(p)
            df = normalize_zeek_conn(df_raw)
            df["source_file"] = p
            frames.append(df)
            print(f"[OK] USTC conn {p}: rows={len(df)}")
        except Exception as e:
            print(f"[WARN] USTC {p}: {e}")

    if not frames:
        raise FileNotFoundError("No input flows found. Check your globs.")
    flows = pd.concat(frames, ignore_index=True)
    return flows.replace([np.inf, -np.inf], np.nan)

# ---------------- metric helpers ----------------
def compute_volume(flows: pd.DataFrame, n_bins: int = BINS_LOG10) -> Dict:
    out = {}
    for col, key in [("bytes","bytes_per_flow"), ("pkts","pkts_per_flow"), ("duration","flow_duration")]:
        edges, cdf = _log10_hist_cdf(flows[col], n_bins)
        if edges:
            out[key] = {"bins_log10": edges, "cdf": cdf}
    return out

def compute_iat(flows: pd.DataFrame, n_bins: int = BINS_LOG10):
    ts = _to_num(flows["start_ts"]).dropna()
    if ts.empty: return None
    ts = ts.sort_values()
    iat = ts.diff().dropna()
    iat = iat[iat > 0]
    if iat.empty: return None
    edges, cdf = _log10_hist_cdf(iat, n_bins)
    if not edges: return None
    return {"bins_log10": edges, "cdf": cdf}

def compute_flow_iat_mean_proxy(flows: pd.DataFrame, n_bins=BINS_LOG10):
    if not {"duration","pkts"}.issubset(flows.columns):
        return None
    dur = _to_num(flows["duration"]).clip(lower=0)
    pk  = _to_num(flows["pkts"]).clip(lower=0)
    denom = (pk.where(pk>=2, 2) - 1)
    mean_iat = (dur / denom).replace([np.inf,-np.inf], np.nan).dropna()
    if mean_iat.empty:
        return None
    edges, cdf = _log10_hist_cdf(mean_iat, n_bins)
    if not edges:
        return None
    return {"bins_log10": edges, "cdf": cdf}

def compute_burstiness(flows: pd.DataFrame, window_s: int = BURST_WINDOW_S):
    ts = _to_num(flows["start_ts"]).dropna()
    if ts.empty: return None
    t0 = ts.min()
    buckets = np.floor((ts - t0) / window_s).astype(int)
    counts = pd.Series(1, index=buckets).groupby(level=0).sum().astype(float)
    lam = counts.mean(); var = counts.var(ddof=0)
    if lam <= 0: return None
    return {"window_s": int(window_s), "fano_ref": float(var/lam)}

def compute_diurnal(flows: pd.DataFrame):
    ts = _to_num(flows["start_ts"]).dropna()
    if ts.empty: return None
    hours = pd.to_datetime(ts, unit="s", utc=True).dt.hour
    counts = hours.value_counts().reindex(range(24), fill_value=0).astype(float)
    total = counts.sum()
    if total <= 0: return None
    return (counts / total).tolist()

def proto_mix(flows: pd.DataFrame) -> Dict[str,float]:
    if "proto" not in flows.columns: return {"unknown": 1.0}
    norm = flows["proto"].astype(str).map(_normalize_proto)
    counts = norm.value_counts()
    if counts.sum() == 0: return {"unknown": 1.0}
    probs = (counts / counts.sum()).to_dict()
    return {str(k): float(v) for k,v in probs.items()}

def service_mix(flows: pd.DataFrame, top_k: int = 20):
    if "service" not in flows.columns: return None
    s = flows["service"].dropna().astype(str).str.lower()
    s = s[(s != "-") & (s != "none") & (s != "")]
    if s.empty: return None
    counts = s.value_counts()
    if len(counts) > top_k:
        top = counts.head(top_k)
        other = counts.iloc[top_k:].sum()
        counts = pd.concat([top, pd.Series({"other": other})])
    probs = (counts / counts.sum()).to_dict()
    return {str(k): float(v) for k,v in probs.items()}

def temporal_by_service(flows: pd.DataFrame,
                        top_k_services: int = 15,
                        min_rows_per_service: int = 200,
                        n_bins: int = BINS_LOG10):
    if "service" not in flows.columns:
        return None
    svc_all = flows["service"].astype(str).str.lower()
    valid = ~svc_all.isin(["-", "none", "", "nan"])
    if not valid.any(): return None
    svc_counts = svc_all[valid].value_counts()
    if svc_counts.empty: return None
    top_svcs = list(svc_counts.head(top_k_services).index)
    total_valid = float(valid.sum())
    result = {}
    for svc in top_svcs:
        mask = (svc_all == svc)
        if mask.sum() < min_rows_per_service:
            continue
        block = flows.loc[mask]
        node = {"count": int(mask.sum()), "share": float(mask.sum() / total_valid)}
        # iat flows
        if "start_ts" in block.columns:
            ts = pd.to_numeric(block["start_ts"], errors="coerce").dropna().sort_values()
            if len(ts) > 1:
                iat = ts.diff().dropna()
                iat = iat[iat > 0]
                if not iat.empty:
                    bins, cdf = _log10_hist_cdf(iat, n_bins)
                    if bins:
                        node["iat_flows"] = {"bins_log10": bins, "cdf": cdf}
        # within-flow mean iat
        vals = None
        if "flow_iat_mean_vals" in block.columns:
            vals = pd.to_numeric(block["flow_iat_mean_vals"], errors="coerce").dropna()
        if (vals is None or vals.empty) and {"duration", "pkts"}.issubset(block.columns):
            dur = pd.to_numeric(block["duration"], errors="coerce").clip(lower=0)
            pk  = pd.to_numeric(block["pkts"], errors="coerce").clip(lower=0)
            denom = (pk.where(pk >= 2, 2) - 1)
            vals = (dur / denom).replace([np.inf, -np.inf], np.nan).dropna()
        if vals is not None and not vals.empty:
            bins, cdf = _log10_hist_cdf(vals, n_bins)
            if bins:
                node["flow_iat_mean"] = {"bins_log10": bins, "cdf": cdf}
        if any(k in node for k in ("iat_flows", "flow_iat_mean")):
            result[svc] = node
    return result or None

# ---------------- timestamp helpers ----------------
def detect_and_normalize_timestamps(flows: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """Detect unit (s / ms / us) from q90 and scale start_ts to seconds if needed.
       Return (flows, diagnostics)."""
    diag = {}
    if "start_ts" not in flows.columns:
        diag["note"] = "no_start_ts"
        return flows, diag
    s = _to_num(flows["start_ts"]).dropna()
    if s.empty:
        diag["note"] = "no_numeric_start_ts"
        return flows, diag
    q90 = float(np.quantile(s.values, 0.90))
    diag["q90"] = q90
    # heuristics:
    #   - if q90 > 1e12 -> microseconds
    #   - if q90 > 1e9  -> seconds (modern)
    #   - if q90 > 1e11 -> likely milliseconds? adjust accordingly
    # We'll prefer common breakpoints: <1e5 -> seconds small, 1e8-1e12 ranges.
    if q90 > 1e12:
        scale = 1e6
        note = "microseconds -> scaling /1e6 applied"
    elif q90 > 1e11:
        # sometimes ms stored but large; treat as ms
        scale = 1e3
        note = "milliseconds -> scaling /1e3 applied"
    elif q90 > 1e9:
        scale = 1.0
        note = "seconds (no scaling)"
    elif q90 > 1e6:
        scale = 1.0  # odd mid-range, keep as-is but warn
        note = "intermediate-range -> no scaling applied (inspect)"
    else:
        scale = 1.0
        note = "small epoch-like values -> no scaling, inspect"
    # apply scaling if !=1
    if scale != 1.0:
        flows["start_ts"] = _to_num(flows["start_ts"]) / scale
    else:
        flows["start_ts"] = _to_num(flows["start_ts"])
    diag["scale"] = scale
    diag["note"] = note
    # compute coverage hours
    ts = flows["start_ts"].dropna()
    if not ts.empty:
        coverage_hours = float((ts.max() - ts.min()) / 3600.0)
        diag["coverage_hours"] = coverage_hours
        diag["min_ts"], diag["max_ts"] = float(ts.min()), float(ts.max())
    return flows, diag

def filter_time_window(flows: pd.DataFrame, min_year=MIN_YEAR, max_year=MAX_YEAR) -> Tuple[pd.DataFrame, Dict]:
    """Filter flows to [min_year..max_year] inclusive using start_ts in seconds."""
    diag = {}
    if "start_ts" not in flows.columns:
        diag["filtered"] = 0
        return flows, diag
    ts = _to_num(flows["start_ts"]).dropna()
    if ts.empty:
        diag["filtered"] = 0
        return flows, diag
    min_ts = datetime(min_year,1,1,tzinfo=timezone.utc).timestamp()
    max_ts = datetime(max_year,12,31,23,59,59,tzinfo=timezone.utc).timestamp()
    mask_valid = (ts >= min_ts) & (ts <= max_ts)
    n_before = len(flows)
    flows_filtered = flows.loc[mask_valid.index.where(mask_valid, False).astype(bool)]
    # simpler: use boolean mask aligned to index
    mask_idx = mask_valid.reindex(flows.index, fill_value=False)
    flows_filtered = flows.loc[mask_idx]
    diag["rows_before"] = n_before
    diag["rows_after"] = len(flows_filtered)
    diag["rows_filtered_out"] = n_before - len(flows_filtered)
    return flows_filtered, diag

# ---------------- build baseline ----------------
def build_baseline(flows: pd.DataFrame) -> Dict:
    notes_local = {}
    # detect/normalize ts units
    flows, diag_ts = detect_and_normalize_timestamps(flows)
    notes_local.update(diag_ts)

    # filter to sane time window
    flows, diag_filter = filter_time_window(flows, MIN_YEAR, MAX_YEAR)
    notes_local.update(diag_filter)

    # canonicalize service labels
    if "service" in flows.columns:
        flows["service"] = canonicalize_service_for_baseline(flows["service"])
        notes_local["service_normalization"] = "applied canonicalize_service_for_baseline"

    # compute coverage_hours if possible
    coverage_hours = None
    if "start_ts" in flows.columns:
        ts = _to_num(flows["start_ts"]).dropna()
        if not ts.empty:
            coverage_hours = float((ts.max() - ts.min()) / 3600.0)

    # metrics
    volume = compute_volume(flows, n_bins=BINS_LOG10)
    iat    = compute_iat(flows, n_bins=BINS_LOG10)
    flow_iat_mean = compute_flow_iat_mean_proxy(flows, n_bins=BINS_LOG10)
    burst  = compute_burstiness(flows, window_s=BURST_WINDOW_S)
    diurn  = compute_diurnal(flows)
    p_mix  = proto_mix(flows)
    s_mix  = service_mix(flows)
    svc_temporal = temporal_by_service(flows, top_k_services=15, min_rows_per_service=200, n_bins=BINS_LOG10)

    metrics = {}
    metrics.update(volume)
    if iat is not None: metrics["iat_flows"] = iat
    if flow_iat_mean is not None: metrics["flow_iat_mean"] = flow_iat_mean
    if burst is not None: metrics["burstiness"] = burst
    if diurn is not None: metrics["diurnal_shape_hourly"] = diurn
    if p_mix: metrics["protocol_mix"] = p_mix
    if s_mix is not None: metrics["service_mix"] = s_mix
    if svc_temporal is not None: metrics["temporal_by_service"] = svc_temporal

    now = datetime.utcnow()
    baseline = {
        "profile": PROFILE_NAME,
        "created_at_utc": now.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "version": now.strftime("%Y-%m"),
        "sources": {
            "mawi_glob": MAWI_GLOB,
            "cic_glob": CIC_GLOB,
            "ustc_glob": USTC_GLOB
        },
        "stats": {
            "n_flows": int(len(flows)),
            "coverage_hours": coverage_hours,
        },
        "metrics": metrics,
        "notes": {
            "winsorize_quantiles": [WINSOR[0], WINSOR[1]],
            "bins": BINS_LOG10,
            "burst_window_s": BURST_WINDOW_S,
            **notes_local
        },
    }
    return baseline

# ---------------- main run ----------------
if __name__ == "__main__":
    print("Loading MAWI + CIC-IoT2023 + USTC (benign-only) sources…")
    flows = load_encrypted_sources(MAWI_GLOB, CIC_GLOB, USTC_GLOB)
    print("Loaded flows:", len(flows))

    baseline = build_baseline(flows)

    # quick sanity printout
    print("\nBaseline profile:", baseline["profile"])
    print("Created:", baseline["created_at_utc"])
    print("Metrics keys:", sorted(list(baseline["metrics"].keys())))
    print("n_flows (stat):", baseline["stats"]["n_flows"])
    print("coverage_hours (stat):", baseline["stats"]["coverage_hours"])

    # save YAML
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_path = os.path.join(OUTPUT_DIR, f"{PROFILE_NAME}_baseline_{datetime.utcnow().strftime('%Y%m%d')}.yaml")
    with open(out_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(baseline, f, sort_keys=False, allow_unicode=True)

    print(f"\nBaseline written to: {out_path}")
    print("✅ Baseline saved. You can now use this file as the 'encrypted_flow_only_ids' baseline in your TRI/DAF pipeline.")


Loading MAWI + CIC-IoT2023 + USTC (benign-only) sources…
[OK] MAWI E:/data/mawi/conn\202508011400.pcap\conn.log.gz: rows=2189082
[OK] MAWI E:/data/mawi/conn\202508031400.pcap\conn.log.gz: rows=8107268
[OK] CIC-IoT CSV E:/encrypted flow/CIC-IoT2023\BenignTraffic1\conn.log.gz: rows=73127
[OK] CIC-IoT CSV E:/encrypted flow/CIC-IoT2023\BenignTraffic2\conn.log.gz: rows=76534
[OK] CIC-IoT CSV E:/encrypted flow/CIC-IoT2023\BenignTraffic3\conn.log.gz: rows=32278
[OK] CIC-IoT CSV E:/encrypted flow/CIC-IoT2023\BenignTraffic\conn.log.gz: rows=160767
[OK] USTC conn E:/encrypted flow/USTC-TFC2016\BitTorrent\conn.log.gz: rows=7517
[OK] USTC conn E:/encrypted flow/USTC-TFC2016\FTP\conn.log.gz: rows=101037
[OK] USTC conn E:/encrypted flow/USTC-TFC2016\Gmail\conn.log.gz: rows=8629
[OK] USTC conn E:/encrypted flow/USTC-TFC2016\MySQL\conn.log.gz: rows=86089
[OK] USTC conn E:/encrypted flow/USTC-TFC2016\Outlook\conn.log.gz: rows=7524
[OK] USTC conn E:/encrypted flow/USTC-TFC2016\Skype\conn.log.gz: rows=63

C:\Users\ifrah\AppData\Local\Temp\ipykernel_32944\3185042802.py:437: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flows["service"] = canonicalize_service_for_baseline(flows["service"])



Baseline profile: encrypted_flow_only_ids
Created: 2025-11-29T08:34:24Z
Metrics keys: ['burstiness', 'bytes_per_flow', 'diurnal_shape_hourly', 'flow_duration', 'flow_iat_mean', 'iat_flows', 'pkts_per_flow', 'protocol_mix', 'service_mix', 'temporal_by_service']
n_flows (stat): 10639056
coverage_hours (stat): 24731.80024107728

Baseline written to: baselines\encrypted_flow_only_ids_baseline_20251129.yaml
✅ Baseline saved. You can now use this file as the 'encrypted_flow_only_ids' baseline in your TRI/DAF pipeline.
